# Week 34, Day 1: Cohesion and coupling

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w034d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
basket = []

def add(item):
    basket.append(item)
    return len(basket)

def checkout_for(name):
    print(name, "pays for", basket)

add("rice")
checkout_for("Ada")
add("yam")
checkout_for("Bo")

In [ ]:
# Low cohesion: one function that parses, calculates, formats and prints.
def process(lines):
    total = 0
    for line in lines:
        name, pence = line.split(",")
        total += int(pence)
    vat = total * 20 // 100
    print(f"Net £{total / 100:.2f}, VAT £{vat / 100:.2f}")

# High cohesion: each function has one job, and one reason to change.
def parse(lines):
    return [(name, int(pence)) for name, pence in (line.split(",") for line in lines)]

def vat_on(total, rate=20):
    return total * rate // 100

def summary(net, vat):
    return f"Net £{net / 100:.2f}, VAT £{vat / 100:.2f}"

lines = ["rice,450", "oil,900"]
process(lines)
items = parse(lines)
net = sum(p for _, p in items)
print(summary(net, vat_on(net)))

In [ ]:
class Address:
    def __init__(self, street, postcode):
        self.street, self.postcode = street, postcode

class Customer:
    def __init__(self, name, address):
        self.name, self.address = name, address

class Order:
    def __init__(self, code, customer):
        self.code, self.customer = code, customer

    def postcode(self):                               # Order answers the question itself
        return self.customer.address.postcode

o = Order("A17", Customer("Ada", Address("12 Broad St", "LS1 4AP")))
print(o.customer.address.postcode)                    # the caller knows Order, Customer AND Address
print(o.postcode())                                   # the caller only knows Order

In [ ]:
class Basket:
    def __init__(self, owner):
        self.owner = owner
        self.items = []

    def add(self, item):
        self.items.append(item)
        return len(self.items)

def checkout(basket):
    print(basket.owner, "pays for", basket.items)

ada, bo = Basket("Ada"), Basket("Bo")
ada.add("rice")
bo.add("yam")
checkout(ada)
checkout(bo)

In [ ]:
import io

# Pure: no printing, no files. Easy to test with plain values.
def overdue(loans, today):
    return sorted(title for title, due in loans if due < today)

def notice(name, titles):
    if not titles:
        return f"Dear {name}, nothing is overdue."
    return f"Dear {name}, please return: " + ", ".join(titles) + "."

# A thin layer of input and output around them.
def send_notice(name, loans, today, out):
    out.write(notice(name, overdue(loans, today)) + "\n")

loans = [("Aké", 20261101), ("Kindred", 20261120), ("Small Island", 20261028)]
print(overdue(loans, 20261103))
buffer = io.StringIO()
send_notice("Ada", loans, 20261103, buffer)
print(buffer.getvalue(), end="")

> **Key idea.** Cohesion is how well the contents of one part belong together; coupling is how much parts depend on each other's details. Aim for high cohesion (each function, class and module has one job you can name without "and") and loose coupling (parts use each other's methods and arguments, not their insides or shared globals). Keep calculation in pure functions, separate from input and output, with a thin shell around a testable core. Ask objects for what you need (`order.postcode()`) rather than reaching through them. Flag arguments, long parameter lists, "and" in names and changing globals are signs to look again.

### Exercise: Calculation without printing

`report` mixes calculating with printing, so it can't be tested or reused. Split it into a pure function `summarise(marks)`, which returns a tuple `(count, average, top)` (average rounded to 1 place; `(0, None, None)` for no marks), and keep `report(marks)` as a thin function that prints the same text as before using `summarise`.

In [ ]:
def report(marks):
    if not marks:
        print("No marks")
        return
    average = round(sum(marks) / len(marks), 1)
    print(f"{len(marks)} marks, average {average}, top {max(marks)}")

report([70, 81, 64])
report([])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_split():\n    \"pure summary, same report\"\n    import io, contextlib\n    assert summarise([70, 81, 64]) == (3, 71.7, 81) and summarise([]) == (0, None, None) and summarise([50]) == (1, 50.0, 50)\n    buf = io.StringIO()\n    with contextlib.redirect_stdout(buf):\n        result = summarise([1, 2])\n    assert buf.getvalue() == \"\", \"summarise shouldn't print anything\"\n    expect_output(\"3 marks, average 71.7, top 81\\nNo marks\")")

### Exercise: Ask, don't reach

`label` reaches through an order into its customer's address. Give `Order` two methods, `postcode()` and `recipient()` (the customer's name), and rewrite `label` to use only those, so that it no longer mentions `customer` or `address`.

In [ ]:
class Address:
    def __init__(self, street, postcode):
        self.street, self.postcode = street, postcode

class Customer:
    def __init__(self, name, address):
        self.name, self.address = name, address

class Order:
    def __init__(self, code, customer):
        self.code, self.customer = code, customer

def label(order):
    return f"{order.customer.name} / {order.customer.address.postcode} / {order.code}"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_label():\n    \"no reaching through\"\n    import ast\n    A, C, O = ns[\"Address\"], ns[\"Customer\"], ns[\"Order\"]\n    o = O(\"A17\", C(\"Ada\", A(\"12 Broad St\", \"LS1 4AP\")))\n    assert label(o) == \"Ada / LS1 4AP / A17\" and o.postcode() == \"LS1 4AP\" and o.recipient() == \"Ada\"\n    tree = ast.parse(source)\n    code = next(ast.get_source_segment(source, n) for n in ast.walk(tree) if isinstance(n, ast.FunctionDef) and n.name == \"label\")\n    assert \"customer\" not in code and \"address\" not in code, \"label should only ask the order\"")

### Exercise: Two functions, not a flag

`price_text(pence, html)` does two different jobs depending on a flag. Replace it with `price_plain(pence)` (like `£12.50`) and `price_html(pence)` (like `<span class="price">£12.50</span>`), sharing one helper so the money formatting is written once. Then update `shelf_label(name, pence)` (plain) and `web_row(name, pence)` (HTML, like `<li>Rice <span class="price">£4.50</span></li>`) to use the new functions, and delete `price_text`.

In [ ]:
def price_text(pence, html):
    if html:
        return f'<span class="price">£{pence // 100}.{pence % 100:02}</span>'
    return f"£{pence // 100}.{pence % 100:02}"

def shelf_label(name, pence):
    return f"{name}: {price_text(pence, False)}"

def web_row(name, pence):
    return f"<li>{name} {price_text(pence, True)}</li>"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_split():\n    \"no flag\"\n    assert price_plain(1250) == \"\u00a312.50\" and price_plain(7) == \"\u00a30.07\"\n    assert price_html(1250) == '<span class=\"price\">\u00a312.50</span>'\n    assert shelf_label(\"Rice\", 450) == \"Rice: \u00a34.50\" and web_row(\"Rice\", 450) == '<li>Rice <span class=\"price\">\u00a34.50</span></li>'\n    assert \"price_text\" not in ns, \"delete price_text\"\n    assert source.count(\"// 100\") == 1, \"write the money formatting once\"")

### Exercise: A long parameter list

`book_class` takes eight arguments, three of which always travel together as a member's contact details. Introduce a frozen dataclass `Contact(name, email, phone)` and change `book_class` to `book_class(contact, gym, day, time, minutes)`, returning the same text as before. Also give `Contact` a method `short()`, giving the name and email like `Ada <ada@example.org>`, and use it in `book_class`.

In [ ]:
from dataclasses import dataclass

def book_class(name, email, phone, gym, day, time, minutes, note=""):
    return f"{name} <{email}> booked {gym}, {day} {time} for {minutes} min"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_contact():\n    \"a parameter object\"\n    import dataclasses, inspect\n    C = ns[\"Contact\"]\n    assert dataclasses.is_dataclass(C) and [f.name for f in dataclasses.fields(C)] == [\"name\", \"email\", \"phone\"]\n    c = C(\"Ada\", \"ada@example.org\", \"0113 496 0000\")\n    assert c.short() == \"Ada <ada@example.org>\"\n    assert book_class(c, \"Leeds Central\", \"Tue\", \"18:00\", 45) == \"Ada <ada@example.org> booked Leeds Central, Tue 18:00 for 45 min\"\n    assert list(inspect.signature(book_class).parameters) == [\"contact\", \"gym\", \"day\", \"time\", \"minutes\"]\n    try:\n        c.name = \"x\"\n        assert False, \"Contact should be frozen\"\n    except AttributeError:\n        pass")

### Bug hunt: Everyone's basket

Two customers share one global basket, so this prints Ada's rice in Bo's total. Replace the global with a `Basket` class (with `add(item, pence)` and `total()`), so the program prints `Ada 450` then `Bo 900`.

In [ ]:
items = []

def add(item, pence):
    items.append((item, pence))

def total():
    return sum(p for _, p in items)

add("rice", 450)
print("Ada", total())
add("oil", 450)
print("Bo", total())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"separate baskets\"\n    B = ns[\"Basket\"]\n    assert \"items\" not in ns or not isinstance(ns.get(\"items\"), list), \"remove the global list\"\n    a, b = B(), B()\n    a.add(\"x\", 1)\n    assert a.total() == 1 and b.total() == 0\n    expect_output(\"Ada 450\\nBo 900\")")

### Bug hunt: A summary that changes its input

After summarising, the caller's list of marks has been sorted, so the last line prints `[45, 70, 88]` instead of the original order. `median` reaches into data it was only asked to read. Fix it so the program prints `70` and then `[88, 45, 70]`.

In [ ]:
def median(marks):
    marks.sort()
    return marks[len(marks) // 2]

scores = [88, 45, 70]
print(median(scores))
print(scores)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"unchanged\"\n    expect_output(\"70\\n[88, 45, 70]\")")

### Bug hunt: The setting that changed under it

`apply_discount` reads a global rate that another function changes, so the second order gets the student discount too, and this prints `900` twice. Make the rate an argument (default 0), remove the global, and fix the calls so it prints `900` then `1000`.

In [ ]:
rate = 0

def student_mode():
    global rate
    rate = 10

def apply_discount(pence):
    return pence - pence * rate // 100

student_mode()
print(apply_discount(1000))
print(apply_discount(1000))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"no shared rate\"\n    import inspect\n    expect_output(\"900\\n1000\")\n    assert \"global\" not in source and \"rate\" in inspect.signature(apply_discount).parameters\n    assert apply_discount(500) == 500 and apply_discount(500, 20) == 400")

### Puzzle: A pure core and a thin shell

Arrange the lines so that the calculation is a pure function and printing happens only in the shell. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Which is better?

For each pair, decide which design has higher cohesion or looser coupling. Answer `"A"` or `"B"`.

In [ ]:
# p: A) def save_and_email(report): ...      B) def save(report): ... and def email(report): ...
# q: A) order.customer.address.town          B) order.town()
# r: A) def total(prices): return sum(prices)          B) def total(prices): print(sum(prices))
# s: A) VAT_RATE = 20 used by several functions        B) the number 20 written in several functions
# t: A) def area(shape, is_circle): ...                B) def circle_area(r): ... and def rect_area(w, h): ...

p = ___
q = ___
r = ___
s = ___
t = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r) == (\"B\", \"B\", \"A\"), \"p to r\"\n    assert (s, t) == (\"A\", \"B\")")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: untangling a script

`run(text)` reads CSV text of gym visits (`member,date,minutes`), works out each member's total minutes and number of visits, picks out members who visited at least 3 times, and prints a report. It does all of that in one function. Rewrite it as a cohesive set of functions with these exact names and jobs, keeping `run` printing exactly the same report:

- `parse_visits(text)`: a list of `(member, date, minutes)` tuples (minutes as `int`), skipping the header; a line without exactly three fields or with minutes that aren't a whole number raises `ValueError`. Pure.
- `totals(visits)`: a dictionary from member to `(visits, minutes)`. Pure.
- `regulars(summary, at_least=3)`: the members with at least that many visits, sorted. Pure.
- `report_lines(summary, regular)`: the report's lines as a list of strings. Pure.
- `run(text)`: the thin shell, which uses the four and prints the lines.

None of the pure functions may print.

In [ ]:
def run(text):
    lines = text.strip().splitlines()[1:]
    summary = {}
    for line in lines:
        parts = line.split(",")
        if len(parts) != 3 or not parts[2].strip().isdigit():
            raise ValueError(f"bad line: {line!r}")
        member, date, minutes = parts[0].strip(), parts[1].strip(), int(parts[2])
        visits, mins = summary.get(member, (0, 0))
        summary[member] = (visits + 1, mins + minutes)
    regular = sorted(m for m, (v, _) in summary.items() if v >= 3)
    for member in sorted(summary):
        v, mins = summary[member]
        star = " *" if member in regular else ""
        print(f"{member}: {v} visits, {mins} min{star}")
    print(f"Regulars: {', '.join(regular) if regular else 'none'}")

DATA = """member,date,minutes
Ada,2026-11-02,45
Bo,2026-11-02,30
Ada,2026-11-03,60
Ada,2026-11-05,30
Chi,2026-11-05,90
Bo,2026-11-06,40
"""
run(DATA)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("DATA = \"\"\"member,date,minutes\nAda,2026-11-02,45\nBo,2026-11-02,30\nAda,2026-11-03,60\nAda,2026-11-05,30\nChi,2026-11-05,90\nBo,2026-11-06,40\n\"\"\"\ndef test_functions():\n    \"the pure functions\"\n    import io, contextlib\n    buf = io.StringIO()\n    with contextlib.redirect_stdout(buf):\n        v = parse_visits(DATA)\n        s = totals(v)\n        r = regulars(s)\n        lines = report_lines(s, r)\n    assert buf.getvalue() == \"\", \"only run should print\"\n    assert v[0] == (\"Ada\", \"2026-11-02\", 45) and len(v) == 6\n    assert s == {\"Ada\": (3, 135), \"Bo\": (2, 70), \"Chi\": (1, 90)} and r == [\"Ada\"] and regulars(s, 2) == [\"Ada\", \"Bo\"]\n    assert lines == [\"Ada: 3 visits, 135 min *\", \"Bo: 2 visits, 70 min\", \"Chi: 1 visits, 90 min\", \"Regulars: Ada\"]\n    assert report_lines({}, []) == [\"Regulars: none\"] and totals([]) == {}\n    for bad in [\"member,date,minutes\\nAda,2026-11-02\\n\", \"member,date,minutes\\nAda,2026-11-02,lots\\n\"]:\n        try:\n            parse_visits(bad)\n            assert False, \"should raise ValueError\"\n        except ValueError:\n            pass\ndef test_run():\n    \"the same report\"\n    expect_output(\"Ada: 3 visits, 135 min *\\nBo: 2 visits, 70 min\\nChi: 1 visits, 90 min\\nRegulars: Ada\")")

## Warm-up

Bo paid for Ada's rice too. Both customers' code reached into the same global list, so each depends on what the other did, in a way nothing in the function signatures reveals. That hidden dependency is **coupling** at its worst, and today is about seeing it and removing it.

## Easy to change

Programs are read and changed far more often than they're written. A useful test of any design is a question: **"if this requirement changed, how many places would I have to edit, and how sure could I be that nothing else broke?"**

- If the VAT rate changes, is it in one place or twelve?
- If the receipts must be emailed instead of printed, does the calculation code have to change?
- If you rename an attribute in `Customer`, how many other classes break?

Two properties of a design answer those questions:

| Property | Meaning | You want it |
|---|---|---|
| **Cohesion** | how closely the things inside one function, class or module belong together | high: each part does one job |
| **Coupling** | how much one part depends on the inner details of another | loose: parts know as little about each other as possible |

They work together. When each part does one job (high cohesion), a change usually lands in one part; when parts depend only on each other's public, stable surface (loose coupling), that change doesn't ripple outwards. The ideas come from the "structured design" work of Larry Constantine and Edward Yourdon in the 1970s, and they're as true for Python classes as they were then.

## Cohesion

A part is **cohesive** when everything in it serves one purpose. You can usually describe it in a short sentence without "and". Compare these:

Both print the same, but the second version can be **reused** (the VAT rule is useful on its own), **tested** piece by piece (Week 24), and **changed** safely: a new receipt layout only touches `summary`.

The same applies to classes and modules. A `Customer` class that also knows how to send emails and write to a database has low cohesion: it will change whenever any of those three things changes. A module called `utils.py` that holds a date formatter, a password checker and a CSV reader isn't cohesive either: it's just a drawer for leftovers.

Cohesion doesn't mean "tiny". A class with twenty methods that are all about one thing (a `Matrix`, say) is cohesive. Five unrelated one-line functions in one class are not.

## Coupling

**Coupling** is how much one part needs to know about another's insides. Some coupling is unavoidable (code that uses an `Account` depends on `Account`), but there are degrees, from loose to tight:

| Looser | Tighter |
|---|---|
| calls a method: `order.postcode()` | reaches through: `order.customer.address.postcode` |
| gets what it needs as arguments | reads and changes global variables |
| depends on a small interface (Week 32's Protocols) | depends on a concrete class's private attributes |
| gets a copy, or a read-only view | shares a mutable list and changes it |

Reaching through objects is worth a closer look:

The first line couples the caller to three classes and to how they're connected. If addresses move from customers to orders (say, for gifts), every such line breaks. The second line asks `Order` a question, and only `Order` needs to change. A common rule of thumb, called the **Law of Demeter**: a method should talk to its own attributes, its arguments and objects it creates, not to the objects *those* return. It's a guideline, not a law: chains on simple data (`config["db"]["host"]`) or on fluent APIs (`text.strip().lower()`) are fine.

**Global state** is the tightest coupling of all, because any code anywhere might read or change it, as the warm-up showed. Pass values in as arguments, or keep state in an object that's passed to whoever needs it:

Module-level **constants** (`VAT_RATE = 20`) are fine: they never change, so nothing can surprise anyone.

## Calculation apart from input and output

The most useful single move for cohesion and coupling is to keep **calculation** separate from **input and output** (reading files, printing, asking the user, the network). Functions that only calculate, taking arguments and returning results without printing, reading or changing anything outside, are called **pure functions**. They're easy to test, easy to reuse, and can't be broken by a missing file.

The program's "shape" becomes a **core** of pure functions and classes that make decisions, wrapped in a thin **shell** that talks to the outside world. Bugs in the logic can be found with simple tests; changes to the outside world (a file instead of the screen, an email instead of a file) only touch the shell. And `send_notice` takes `out` as an argument, so it works with the screen, a file or `io.StringIO`: Week 30's duck typing, used to loosen coupling.

## Warning signs

Some code smells point straight at weak cohesion or tight coupling:

| Sign | Example | Usually means |
|---|---|---|
| "and" in a name | `parse_and_save`, `validate_and_send` | two jobs in one function |
| a flag argument | `format(price, as_html=True)` | two functions sharing a body |
| a long parameter list | `book(name, email, phone, street, town, postcode, ...)` | some of them belong together in an object |
| reaching through | `a.b.c.d` | the caller knows too much about the structure |
| a global that changes | `basket = []` at the top, `basket.append` inside functions | hidden coupling between every function that touches it |
| printing inside a calculation | `def total(...): ...; print(...)` | calculation tied to one kind of output |

None of these is automatically wrong, but each deserves a second look. The practice gives you one of each to fix.

## Practice

## Quiz

## Daily challenge

## Before you go

Take the largest function you've written in this course (the Week 25 project is a good place to look) and describe what it does in one sentence. If the sentence needs "and", sketch how you'd split it. Tomorrow: the first two SOLID principles, single responsibility and open-closed, which build directly on today.